# 03. Бустинг для классификации вручную: 8 объектов, логиты и шаг Ньютона

| | |
|---|---|
| **Уровень** | 1 из 4 — крошечные данные: всё считается вручную (`1_tiny`) |
| **Скрипт-источник** | [`03_classification_by_hand.py`](../03_classification_by_hand.py) — тот же код, запуск из терминала |
| **Время выполнения** | ≈ 1 с |

**Цель.** Понять, почему классификатор бустинга работает с логитами и чем отличается шаг Ньютона.

### Чему вы научитесь

- Старт F₀ = log(p / (1 − p))
- Вероятность p = σ(F)
- Градиент g = p − y и гессиан h = p(1 − p)
- Значение листа w = −G / H
- Как падает log-loss
- Сверка с `gbcourse` в режиме `newton`

### Данные

`toy_classification` — 8 объектов, x = 1…8, класс y = (0, 0, 0, 1, 0, 1, 1, 1): классы «перемешаны» посередине, как почти всегда в жизни.

### План

1. **Этап 1.** Данные
2. **Этап 2.** Старт: логит доли класса 1
3. **Этап 3.** Дерево 1: g, h, выигрыш G²/H и значения листьев −G/H
4. **Этап 4.** Деревья 2 и 3
5. **Этап 5.** Сверка с gbcourse.GBClassifier(mode='newton', max_depth=1, 3 дерева, ν = 1)

> Ноутбук собран из `examples/1_tiny/03_classification_by_hand.py` командой `python examples/build_notebooks.py`. Чтобы изменить пример, правьте скрипт и пересоберите ноутбук.

## Подготовка

Ячейка ниже находит папку `examples/` (ноутбук можно открыть из любой рабочей папки внутри
репозитория), подключает общий каркас примеров `_common.Example` и учебную библиотеку курса
`gbcourse`. Каркас печатает этапы и выводы, «раскрывает» датасет и показывает графики прямо в
ноутбуке. Выполняйте ячейки сверху вниз: **Run → Run All Cells**.

In [1]:
import sys
from pathlib import Path

EXAMPLES = next(p / "examples" for p in [Path.cwd(), *Path.cwd().parents]
                if (p / "examples" / "_common.py").is_file())
sys.path.insert(0, str(EXAMPLES))
from _common import Example

ex = Example(EXAMPLES / "1_tiny/03_classification_by_hand.py", title="03. Бустинг для классификации вручную: 8 объектов",
             goal="пройти шаги бустинга для log-loss: логиты, g, h, лист −G/H")

import numpy as np
from gbcourse import GBClassifier, datasets

sigmoid = lambda z: 1 / (1 + np.exp(-z))
logloss = lambda y, p: float(-np.mean(y * np.log(p) + (1 - y) * np.log(1 - p)))

## Этап 1. Данные

In [2]:
X, y = datasets.toy_classification()
x = X[:, 0]
ex.describe(X, y, names=["x"], target="класс", task="classification", source="gbcourse.datasets.toy_classification()")

   Источник: gbcourse.datasets.toy_classification()
   Объектов: 8   признаков: 1   память: 0.00 МБ
   Типы признаков: float64 × 1
   Пропусков нет
   Цель «класс»: класс 0 — 4 (50.0%), класс 1 — 4 (50.0%)
   Первые строки:


,x,[класс]
0,1.0,0
1,2.0,0
2,3.0,0


## Этап 2. Старт: логит доли класса 1

In [3]:
p1 = y.mean()
F = np.full(len(y), np.log(p1 / (1 - p1)))
print(f"   доля класса 1 = {p1}, F₀ = log({p1}/{1 - p1}) = {F[0]:.4f}, p = σ(F₀) = {sigmoid(F[0]):.4f}")
print(f"   log-loss = {logloss(y, sigmoid(F)):.4f} (так же, как у «монетки» 0.5: log 2 = {np.log(2):.4f})")


def one_tree(F, verbose):
    p = sigmoid(F)
    g, h = p - y, p * (1 - p)
    if verbose:
        print("   x:          " + " ".join(f"{v:6.0f}" for v in x))
        print("   y:          " + " ".join(f"{v:6.0f}" for v in y))
        print("   p = σ(F):   " + " ".join(f"{v:6.3f}" for v in p))
        print("   g = p − y:  " + " ".join(f"{v:+6.3f}" for v in g))
        print("   h = p(1−p): " + " ".join(f"{v:6.3f}" for v in h))
    best = None
    for k in range(1, len(x)):
        t = (x[k - 1] + x[k]) / 2
        L, R = x <= t, x > t
        gain = g[L].sum() ** 2 / h[L].sum() + g[R].sum() ** 2 / h[R].sum() - g.sum() ** 2 / h.sum()
        if best is None or gain > best[0]:
            best = (gain, t, -g[L].sum() / h[L].sum(), -g[R].sum() / h[R].sum())
    return best

   доля класса 1 = 0.5, F₀ = log(0.5/0.5) = 0.0000, p = σ(F₀) = 0.5000
   log-loss = 0.6931 (так же, как у «монетки» 0.5: log 2 = 0.6931)


## Этап 3. Дерево 1: g, h, выигрыш G²/H и значения листьев −G/H

In [4]:
gain, t, wl, wr = one_tree(F, verbose=True)
print(f"   лучший пень: x ≤ {t}; листья w = −G/H: {wl:+.4f} и {wr:+.4f}  (выигрыш {gain:.3f})")
F = F + np.where(x <= t, wl, wr)          # темп ν = 1: полный шаг Ньютона
print(f"   log-loss после дерева 1: {logloss(y, sigmoid(F)):.4f}")
ex.note("Лист — это шаг Ньютона: сумма градиентов, делённая на сумму «кривизн» h.")

   x:               1      2      3      4      5      6      7      8
   y:               0      0      0      1      0      1      1      1
   p = σ(F):    0.500  0.500  0.500  0.500  0.500  0.500  0.500  0.500
   g = p − y:  +0.500 +0.500 +0.500 -0.500 +0.500 -0.500 -0.500 -0.500
   h = p(1−p):  0.250  0.250  0.250  0.250  0.250  0.250  0.250  0.250
   лучший пень: x ≤ 3.5; листья w = −G/H: -2.0000 и +1.2000  (выигрыш 4.800)
   log-loss после дерева 1: 0.3621


> ▸ **Вывод.** Лист — это шаг Ньютона: сумма градиентов, делённая на сумму «кривизн» h.

## Этап 4. Деревья 2 и 3

In [5]:
for m in (2, 3):
    gain, t, wl, wr = one_tree(F, verbose=False)
    F = F + np.where(x <= t, wl, wr)
    print(f"   дерево {m}: x ≤ {t}, листья {wl:+.3f} / {wr:+.3f} → log-loss {logloss(y, sigmoid(F)):.4f}")
print("   итоговые вероятности: " + " ".join(f"{v:.2f}" for v in sigmoid(F)))
ex.note("Объекты x = 4 (класс 1) и x = 5 (класс 0) стоят «не на своих местах»: модель менее уверена в них (0.75 и 0.34), чем в остальных (0.11 и 0.88).")

   дерево 2: x ≤ 5.5, листья -1.334 / +1.301 → log-loss 0.2165
   дерево 3: x ≤ 4.5, листья +1.234 / -0.521 → log-loss 0.1800
   итоговые вероятности: 0.11 0.11 0.11 0.75 0.34 0.88 0.88 0.88


> ▸ **Вывод.** Объекты x = 4 (класс 1) и x = 5 (класс 0) стоят «не на своих местах»: модель менее уверена в них (0.75 и 0.34), чем в остальных (0.11 и 0.88).

## Этап 5. Сверка с gbcourse.GBClassifier(mode='newton', max_depth=1, 3 дерева, ν = 1)

In [6]:
lib = GBClassifier(mode="newton", n_estimators=3, max_depth=1, learning_rate=1.0).fit(X, y)
diff = np.abs(lib.predict_proba(X)[:, 1] - sigmoid(F)).max()
print(f"   наибольшее расхождение вероятностей: {diff:.1e}")
assert diff < 1e-10
ex.done()

   наибольшее расхождение вероятностей: 2.2e-16
Готово за 0.2 с


## Попробуйте сами

Измените код в ячейках выше и выполните их заново:

1. Возьмите темп ν = 0.3 вместо полного шага Ньютона и сравните log-loss после трёх деревьев.
2. Замените шаг Ньютона градиентным (лист = −среднее g): насколько медленнее падает log-loss?
3. Добавьте дубликат объекта класса 1 и посмотрите, как изменится стартовый логит F₀.

## Что дальше

- **Теория в курсе:** [6.1 «Логиты и log-loss»](../../../lessons/lesson_6_1/README.md), [6.2 «Шаг Ньютона»](../../../lessons/lesson_6_2/README.md).
- **Предыдущий пример:** [02. Дерево решений вручную: 10 объектов, 2 признака](02_tree_by_hand.ipynb)
- **Следующий пример:** [04. Темп обучения на 20 точках: быстро и переобученно или медленно и точно](04_learning_rate_by_hand.ipynb)
- **Все примеры:** [examples/README.md](../../README.md)